# EDA 02 — AffectiveROAD Dataset
Exploratory analysis of the real-world driving stress dataset.

Signals: E4 wristband (EDA 4Hz, BVP 64Hz, TEMP 4Hz, ACC 32Hz)  
Labels: Continuous subjective stress metric (SM_Drv*.csv) → binarised at 75th percentile  
Drives: Drv1–Drv13 (13 drives)

In [ ]:
import sys
sys.path.insert(0, '..')

import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from io import BytesIO

AR_ROOT = Path('/Users/omarnassar/Documents/Thesis/AffectiveROAD_Data')
E4_DIR  = AR_ROOT / 'Database' / 'E4'
SM_DIR  = AR_ROOT / 'Database' / 'Subj_metric'

In [ ]:
# List available drives
drives = sorted(E4_DIR.glob('*-E4-Drv*'))
print(f'Found {len(drives)} drive folders:')
for d in drives:
    zips = list(d.glob('*.zip'))
    print(f'  {d.name}: {[z.name for z in zips]}')

In [ ]:
# Load E4 signals from Drv1 Left.zip
def parse_e4_csv(raw_bytes, name):
    lines = raw_bytes.decode('utf-8').strip().split('\n')
    if len(lines) < 3:
        return None, None
    ts = float(lines[0].strip())
    fs = float(lines[1].strip())
    vals = np.array([float(x.strip()) for x in lines[2:] if x.strip()])
    return vals, fs

drv1_zip = list(E4_DIR.glob('*Drv1/Left.zip'))
if drv1_zip:
    with zipfile.ZipFile(drv1_zip[0], 'r') as zf:
        files = zf.namelist()
        print('Files in zip:', files)
        signals = {}
        for fname in ['EDA.csv', 'BVP.csv', 'TEMP.csv']:
            if fname in files:
                raw = zf.read(fname)
                vals, fs = parse_e4_csv(raw, fname)
                if vals is not None:
                    signals[fname.replace('.csv', '')] = (vals, fs)
                    print(f'  {fname}: {len(vals)} samples at {fs} Hz')
else:
    print('Drv1 not found — check AR_ROOT path')

In [ ]:
# Plot EDA + stress label for Drv1
sm_file = SM_DIR / 'SM_Drv1.csv'
if sm_file.exists() and 'EDA' in signals:
    sm = pd.read_csv(sm_file, header=None, names=['score'])
    print(f'SM Drv1: {len(sm)} samples, range [{sm.score.min():.2f}, {sm.score.max():.2f}]')

    eda, fs_eda = signals['EDA']
    threshold = np.percentile(sm['score'], 75)
    print(f'Stress threshold (75th pct): {threshold:.2f}')

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

    t_eda = np.arange(len(eda)) / fs_eda
    axes[0].plot(t_eda, eda, lw=0.8, color='teal')
    axes[0].set_ylabel('EDA (μS)')
    axes[0].set_title('Drv1 — EDA signal')

    t_sm = np.arange(len(sm)) / 1.0  # SM is 1 Hz
    axes[1].plot(t_sm, sm['score'], lw=0.8, color='coral')
    axes[1].axhline(threshold, color='k', linestyle='--', lw=1, label=f'75th pct ({threshold:.2f})')
    axes[1].fill_between(t_sm, threshold, sm['score'].values,
                          where=sm['score'].values > threshold, alpha=0.3, color='red', label='Stress')
    axes[1].set_ylabel('Stress metric')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('Run script 02_preprocess_affectiveroad.py or check paths.')

In [ ]:
# Check route annotation structure
annot_file = SM_DIR / 'Annot_Subjective_metric.csv'
if annot_file.exists():
    annot = pd.read_csv(annot_file)
    print(annot.head(20))
    print('\nColumns:', list(annot.columns))

In [ ]:
# Check processed windows
processed_path = Path('../data/processed/affectiveroad/Drv1_windows.npz')
if processed_path.exists():
    arr = np.load(processed_path)
    print(f'X shape : {arr["X"].shape}')
    print(f'y shape : {arr["y"].shape}')
    print(f'Stress  : {arr["y"].sum()} / {len(arr["y"])} = {100*arr["y"].mean():.1f}%')
else:
    print('Run script 02_preprocess_affectiveroad.py first.')